In [ ]:
import hydra
from langchain_core.messages import HumanMessage, SystemMessage

from social_groups.trialrunner.config import get_llm
from social_groups.trialrunner.utils.cli_utils import setup_config

%load_ext autoreload
%autoreload 2

In [ ]:
USED_CASE = 2

cases = [
    [
        "+experiment=final/baseline_ministral3",
        "experiment/hetero/backend@experiment.strategy.configuration.backend=ministral3-3b",
    ],
    ["+experiment=final/baseline"],
    ["+experiment=final/baseline_qwen35"],
]

In [ ]:
with hydra.initialize(version_base=None, config_path="../configs/trials"):
    cfg = hydra.compose(
        config_name="local",
        overrides=cases[USED_CASE],
    )

config = setup_config(cfg)

In [ ]:
llm = get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
)

llm

In [ ]:
messages = [
    SystemMessage("You are a professor. Answer the following:"),
    HumanMessage("This is a test. What os 3 + 3? (A) 6 (B) 12 (C) 15"),
    # HumanMessage(
    #     "You have to first think about the correct answer. Then submit your answer using the given tool."
    # ),
]

### With Tool Call

In [ ]:
from social_groups.trialrunner.utils.tool_calls import parse_tool_call_arguments
from langchain_core.tools import tool


@tool(return_direct=True)
def propose_solution(correct_answer: str, reasoning: str):
    """
    Submit the answer to the question that you think is correct.
    Please provide extensive reasoning on why this is correct,
    which assumptions and knowledge was used to retrieve the answer
    and what argumentation is needed to come to the conclusion.

    :arg correct_answer: The correct answer-letter of the question. in the form: (X)
    :arg reasoning: Fully specified reasoning path to retrieve this answer.
    """
    pass


answer = llm.bind_tools([propose_solution]).invoke(
    messages
)

print(answer)
print()
print(parse_tool_call_arguments(answer)["correct_answer"])

### Standard call

In [ ]:
llm.invoke(messages)

### With Thinking

In [ ]:
get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
    with_thinking=True,
).invoke(messages)

### Without Thinking

In [ ]:
get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
    with_thinking=False,
).invoke(messages)

### With Structured Output

In [ ]:
from pydantic import Field
from pydantic import BaseModel


class AnswerResponseFormat(BaseModel):
    response: str = Field(
        ..., description="The answer to the question in the form 'The answer is (X)'."
    )


get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
    with_thinking=False,
).with_structured_output(
    AnswerResponseFormat,
    strict=True,
    include_raw=True,
    method="json_schema" if "mistral" in config.experiment.strategy.configuration.backend.model_name else "function_calling",
).invoke(messages)